# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamarBabar02/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

### Chosen method: Logistic Regression

My lane is Content Opportunity Scoring / Refresh ranking, so the goal is to assign scores to pages and rank them by their likelihood of being a useful refresh opportunity.

I chose Logistic Regression as the first learned model because it is simple, interpretable, and produces probability scores that can be used to rank items.

This is appropriate as a first model because it provides a transparent learned-model comparison against the Week-4 rule-based baseline. I will evaluate both approaches using the same evaluation rows and the same ranking metrics.

I will only consider a more complex model if the results show that additional complexity provides a meaningful improvement.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I will use a grouped train/test split by client_hash_id.

The starter dataset contains content items from multiple clients. The client identifier is used only for grouping and splitting, not as a model feature.

Grouping by client is an honest split because it prevents content from the same client from appearing in both the training and test sets. This reduces the risk that the model learns client-specific patterns and then receives an artificially easy test set.

The Week-4 baseline did not use a train/test split; it was a March 2026 rule-based ranking. Therefore, in Week 5 I will evaluate that baseline rule on the same held-out test rows used for the Logistic Regression model. This makes the comparison fair without claiming that Week 4 already had a train/test split.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

The target is whether a content item becomes a CTR improvement opportunity.

To avoid using future information, the target is constructed from the observed March 2026 data available in this modeling dataset. The model is trained only on the training clients and evaluated on unseen clients.

The model uses search-performance features that are available at scoring time. Leakage-prone fields such as trend_pct and trend_direction are not used.

The model probability is used as the learned ranking score.

The Week-4 rule-based baseline is also applied to exactly the same test rows. Both methods are evaluated using Precision@20 and Precision@50.

In [5]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score

RANDOM_STATE = 42

In [6]:
# Load the same March 2026 dataset used in Week 4

!pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

rel = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
"""

query = f"""
SELECT *
FROM {rel}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
"""

df = con.sql(query).df()

print("Shape:", df.shape)
print("Start date:", df["report_date"].min())
print("End date:", df["report_date"].max())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (9841378, 31)
Start date: 2026-03-01 00:00:00
End date: 2026-03-31 00:00:00


In [ ]:
# Work on a copy
model_df = df.copy()

# CTR
model_df["ctr"] = np.where(
    model_df["gsc_impressions"] > 0,
    model_df["gsc_clicks"] / model_df["gsc_impressions"],
    np.nan
)

# Valid search position
model_df["valid_position"] = model_df["gsc_avg_position"].where(
    model_df["gsc_avg_position"] > 0
)

# Baseline score from Week 4
model_df["baseline_score"] = (
    (1 - model_df["ctr"].clip(0, 1)) * 0.6
    + (1 / (1 + model_df["valid_position"])) * 0.4
)

# Week-4 rule
model_df["baseline_reason"] = np.where(
    (model_df["ctr"] <= 0.02) &
    (model_df["valid_position"] <= 3),
    "LOW_CTR_STRONG_POSITION",
    "OTHER"
)

model_df["baseline_action"] = np.where(
    model_df["baseline_reason"] == "LOW_CTR_STRONG_POSITION",
    "REVIEW_CTR",
    "NO_ACTION"
)

print(model_df[[
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "baseline_score",
    "baseline_action"
]].head())

In [ ]:
# Target definition:
# 1 = observed CTR improvement opportunity according to the Week-4 rule
# 0 = otherwise

model_df["target"] = (
    (model_df["ctr"] <= 0.02) &
    (model_df["valid_position"] <= 3)
).astype(int)

print("Target distribution:")
display(
    model_df["target"]
    .value_counts()
    .rename(index={0: "Not opportunity", 1: "Opportunity"})
    .to_frame("count")
)

print("\nPositive rate:", round(model_df["target"].mean(), 4))

In [ ]:
# Features available from the observed snapshot.
# IDs are excluded from modeling.
# Leakage-prone trend fields are excluded.

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "scroll_events",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "ai_chatgpt",
    "ai_perplexity",
    "ai_gemini",
    "ai_copilot",
    "ai_claude",
    "ai_meta",
    "ai_other"
]

X = model_df[feature_cols].copy()
y = model_df["target"].copy()
groups = model_df["client_hash_id"].copy()

print("Number of features:", len(feature_cols))
print("Client IDs used as feature:", "client_hash_id" in feature_cols)

In [ ]:
# Grouped split by client.
# The same client cannot appear in both train and test.

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_groups = groups.iloc[train_idx]
test_groups = groups.iloc[test_idx]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print("Train clients:", train_groups.nunique())
print("Test clients:", test_groups.nunique())

print(
    "Client overlap:",
    len(set(train_groups.unique()) & set(test_groups.unique()))
)

In [ ]:
# Logistic Regression pipeline:
# median imputation -> scaling -> logistic regression

logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ))
])

logistic_model.fit(X_train, y_train)

# Probability of being an opportunity
model_scores = logistic_model.predict_proba(X_test)[:, 1]

model_predictions = (model_scores >= 0.5).astype(int)

print("Logistic Regression trained successfully.")

In [ ]:
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_indices = np.argsort(scores)[::-1][:k]

    return y_true[top_indices].mean()

In [ ]:
test_rows = model_df.iloc[test_idx].copy()

baseline_test_scores = test_rows["baseline_score"].to_numpy()
baseline_test_target = test_rows["target"].to_numpy()

model_test_target = y_test.to_numpy()

results = []

for k in [20, 50]:
    baseline_p = precision_at_k(
        baseline_test_target,
        baseline_test_scores,
        k
    )

    model_p = precision_at_k(
        model_test_target,
        model_scores,
        k
    )

    results.append({
        "Metric": f"Precision@{k}",
        "Week-4 Baseline": baseline_p,
        "Logistic Regression": model_p,
        "Difference": model_p - baseline_p
    })

comparison = pd.DataFrame(results)

display(comparison)

In [ ]:
comparison_display = comparison.copy()

for col in [
    "Week-4 Baseline",
    "Logistic Regression",
    "Difference"
]:
    comparison_display[col] = (
        comparison_display[col] * 100
    ).round(2).astype(str) + "%"

display(comparison_display)

In [ ]:
precision = precision_score(
    y_test,
    model_predictions,
    zero_division=0
)

print("Logistic Regression classification precision:", round(precision, 4))

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The model is evaluated on clients that were not present in training.

For ranking, Precision@20 and Precision@50 are more useful than accuracy because the practical question is which pages should be reviewed first.

I will inspect false positives and false negatives to understand where the model makes mistakes. I will also inspect model coefficients to understand which observed features influence the ranking.

The interpretation is directional: feature importance shows what the model used, not proof that a feature causes the outcome.

In [ ]:
error_df = test_rows[[
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "target"
]].copy()

error_df["model_score"] = model_scores
error_df["model_prediction"] = model_predictions

error_df["error_type"] = np.select(
    [
        (error_df["target"] == 1) & (error_df["model_prediction"] == 0),
        (error_df["target"] == 0) & (error_df["model_prediction"] == 1)
    ],
    [
        "False Negative",
        "False Positive"
    ],
    default="Correct"
)

display(
    error_df["error_type"]
    .value_counts()
    .to_frame("count")
)

In [ ]:
false_positives = (
    error_df[
        error_df["error_type"] == "False Positive"
    ]
    .sort_values("model_score", ascending=False)
    .head(10)
)

print("Top false positives:")
display(false_positives)

In [ ]:
false_negatives = (
    error_df[
        error_df["error_type"] == "False Negative"
    ]
    .sort_values("model_score", ascending=True)
    .head(10)
)

print("Top false negatives:")
display(false_negatives)

In [ ]:
# Extract Logistic Regression coefficients.
# The pipeline adds missing-value indicators, so get the final feature names.

imputer = logistic_model.named_steps["imputer"]
model = logistic_model.named_steps["model"]

feature_names = imputer.get_feature_names_out(feature_cols)

coefficients = pd.DataFrame({
    "feature": feature_names,
    "coefficient": model.coef_[0]
})

coefficients["abs_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "abs_coefficient",
    ascending=False
)

print("Top 10 features by absolute coefficient:")
display(
    coefficients.head(10)[
        ["feature", "coefficient"]
    ]
)

### Interpretation

The largest coefficients identify features that the Logistic Regression model relied on most strongly when separating opportunity and non-opportunity rows.

These relationships should be treated as model associations rather than causal effects.

In particular, search visibility and CTR-related variables are expected to be important because the target represents pages with relatively low CTR and strong search position.

In [ ]:
print("Three concrete error cases:")

error_examples = (
    error_df[
        error_df["error_type"] != "Correct"
    ]
    .sort_values("model_score", ascending=False)
    .head(3)
)

display(
    error_examples[
        [
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "gsc_avg_position",
            "model_score",
            "target",
            "model_prediction",
            "error_type"
        ]
    ]
)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.